# JEDI Enhanced Ridge -> XGBoost K-Fold Workbook

**Input:** `jedi_modeling_enhanced_dataset.csv`

**Changes from previous notebook:**

- uses all 691 rows after fixing `experience_profile_fit_score`
- adds explicit rule features
- drops weak engineered features inside each fold
- uses word TF-IDF plus character TF-IDF
- reports classification and ranking metrics

## Step 0: Imports And Paths

In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    mean_absolute_error,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "xgboost is not installed in this kernel. Run '%pip install xgboost', restart the kernel, then rerun."
    ) from exc

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 140)

RANDOM_STATE = 42
N_SPLITS = 5
DROP_LOW_CORR_COUNT = 20


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "jedi_modeling_enhanced_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find jedi_modeling_enhanced_dataset.csv.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "jedi_modeling_enhanced_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "data"

print(PROJECT_ROOT)
print(DATA_PATH)

d:\ML\JEDI
d:\ML\JEDI\jedi_modeling_enhanced_dataset.csv


## Step 1: Load Enhanced Dataset

In [3]:
df = pd.read_csv(DATA_PATH)

print("Rows: {:,}".format(len(df)))
print("Columns: {:,}".format(df.shape[1]))
display(df.head(3))
display(df["applied"].value_counts().rename_axis("applied").reset_index(name="rows"))

if "experience_profile_fit_score" not in df.columns:
    raise ValueError("Enhanced dataset missing experience_profile_fit_score")

print("Experience score NA rows:", df["experience_profile_fit_score"].isna().sum())

Rows: 691
Columns: 90


,row_id,source_platform,source_type,scraped_at,can_apply_now,company_name,company_match_score,role_title,role_title_preference_score,role_family,role_family_preference_score,seniority,seniority_profile_fit_score,location,location_profile_fit_score,work_mode,work_mode_fit_score,employment_type,experience_min_years,experience_max_years,experience_profile_fit_score,skills_or_requirements,skills_profile_fit_score,salary_min,salary_max,salary_profile_fit_score,freshness,overall_preference_score,preference_reason,applied,apply_url,skills_or_requirements_original,skills_array,has_skills_or_requirements,skills_imputation_type,role_family_standard,seniority_standard,employment_type_standard,work_mode_standard,city_standard,country_standard,location_bucket,is_bengaluru,is_gurgaon_ncr,is_india,experience_min_bucket,salary_midpoint,salary_available,near_duplicate_key,company_name_standard,role_title_standard,experience_profile_fit_score_original,experience_profile_fit_imputed,experience_profile_fit_imputation_reason,feat_is_experience_mismatch,feat_is_senior_or_above,feat_is_lead_staff_principal,feat_is_early_career,feat_is_intern_or_new_grad,feat_is_entry_or_junior,feat_is_mid_level,feat_is_backend_or_fullstack,feat_is_ml_ai,feat_is_data_role,feat_is_devops_cloud,feat_is_qa_testing,feat_is_support_or_annotation,feat_is_security_role,feat_is_platform_infra,feat_is_llm_or_rag_role,feat_is_remote,feat_is_remote_global,feat_is_india_remote,feat_is_india_target_location,feat_is_bengaluru_or_ncr,feat_is_tier1_city,feat_is_location_mismatch,feat_is_low_salary,feat_is_mid_salary,feat_is_high_salary,feat_salary_good_but_role_bad,feat_low_salary_but_good_role,feat_good_role_good_location,feat_good_title_bad_experience,feat_remote_early_career,feat_onsite_or_hybrid_early_career,feat_bad_fit_combo,feat_strong_fit_combo,enhanced_title_skill_text,enhanced_context_text
0,1,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,Nirmata,4,"Software Engineer Intern, India",5,Backend / Cloud,4,intern,5,"Bangalore, Hybrid",4,Hybrid,5,internship,0.0,1.0,5,Go; Java; microservices; Kubernetes; Prometheus; TimeScale; reliability; automation,2,1200000,1200000,4,Not visible,3,"Intern role is fine, everything is good but the requirements state Kubernetes, observability tools, promethus and Go which dont match wi...",no,https://job-boards.greenhouse.io/nirmata/jobs/5040355008,Go; Java; microservices; Kubernetes; Prometheus; TimeScale; reliability; automation,"[""go"",""java"",""microservices"",""kubernetes"",""prometheus"",""timescale"",""reliability"",""automation""]",True,original,backend,intern,internship,hybrid,bengaluru,india,bengaluru,True,False,True,0,1200000,True,nirmata|software_engineer_intern_india|bengaluru,nirmata,software_engineer_intern_india,5.0,0,existing_numeric,0,0,0,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,1,1,0,0,1,0,0,0,1,0,0,1,0,1,"Software Engineer Intern, India | Backend / Cloud | backend | intern | Go; Java; microservices; Kubernetes; Prometheus; TimeScale; relia...","Software Engineer Intern, India | Backend / Cloud | intern | Bangalore, Hybrid | Hybrid | internship | Go; Java; microservices; Kubernet..."
1,2,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,CloudSEK,5,SDE - Backend - Intern,5,Backend,5,intern,5,"Bengaluru, Karnataka, India",4,Hybrid,5,internship,0.0,1.0,5,Backend engineering; cybersecurity product; APIs; databases likely,4,800000,900000,3,Not visible,4,"Backend intern in Bengaluru; very relevant for fresher profile, salary is a little less and location bengaluru",yes,https://job-boards.greenhouse.io/cloudsek/jobs/5766402004,Backend engineering; cybersecurity product; APIs; databases likely,"[""backend engineering"",""cybersecurity product"",""apis"",""databases likely""]",True,original,backend,intern,internship,hybrid,bengaluru,india,bengaluru,True,False,True,0,850000,True,cloudsek|sde_backend_intern|bengaluru,cloudsek,sde_backend_intern,5.0,0,existing_numeric,0,0,0,1,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,1,1,0,0,1,0,0,0,1,0

,applied,rows
0,no,386
1,yes,305


Experience score NA rows: 0


## Step 2: Targets

In [4]:
SCORE_TARGETS = [
    "role_title_preference_score",
    "role_family_preference_score",
    "seniority_profile_fit_score",
    "location_profile_fit_score",
    "work_mode_fit_score",
    "experience_profile_fit_score",
    "skills_profile_fit_score",
    "salary_profile_fit_score",
    "overall_preference_score",
]

df["applied_clean"] = df["applied"].astype(str).str.strip().str.lower()
bad_labels = sorted(set(df["applied_clean"]) - {"yes", "no"})
if bad_labels:
    raise ValueError("Invalid applied labels: {}".format(bad_labels))

df["target_applied"] = df["applied_clean"].map({"no": 0, "yes": 1}).astype(int)

for col in SCORE_TARGETS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid_rows = df["target_applied"].notna() & df[SCORE_TARGETS].notna().all(axis=1)
if valid_rows.sum() != len(df):
    print("Rows dropped due to missing target values:", len(df) - valid_rows.sum())

model_df = df.loc[valid_rows].reset_index(drop=True).copy()
y_applied = model_df["target_applied"].astype(int)
y_scores = model_df[SCORE_TARGETS].astype(float)

display(pd.DataFrame({
    "target": SCORE_TARGETS,
    "min": [y_scores[c].min() for c in SCORE_TARGETS],
    "max": [y_scores[c].max() for c in SCORE_TARGETS],
    "mean": [y_scores[c].mean() for c in SCORE_TARGETS],
    "missing": [y_scores[c].isna().sum() for c in SCORE_TARGETS],
}))
display(y_applied.value_counts().rename_axis("target_applied").reset_index(name="rows"))

,target,min,max,mean,missing
0,role_title_preference_score,0.0,5.0,4.114327,0
1,role_family_preference_score,1.0,5.0,4.218524,0
2,seniority_profile_fit_score,0.0,5.0,3.561505,0
3,location_profile_fit_score,0.0,5.0,3.684515,0
4,work_mode_fit_score,1.0,5.0,4.360347,0
5,experience_profile_fit_score,0.0,5.0,3.641100,0
6,skills_profile_fit_score,1.0,5.0,3.845152,0
7,salary_profile_fit_score,0.0,8.0,4.215630,0
8,overall_preference_score,0.0,5.0,3.266281,0


,target_applied,rows
0,0,386
1,1,305


## Step 3: Leakage-Free Features

In [5]:
ALL_SCORE_COLUMNS = [
    c for c in model_df.columns
    if c.endswith("_score") or c in ["company_match_score", "overall_preference_score"]
]

LEAKAGE_OR_METADATA_COLUMNS = [
    "row_id",
    "applied",
    "applied_clean",
    "target_applied",
    "preference_reason",
    "apply_url",
    "source_platform",
    "source_type",
    "scraped_at",
    "near_duplicate_key",
    "company_name",
    "company_name_standard",
    "role_title_standard",
]

drop_columns = sorted(set(ALL_SCORE_COLUMNS + LEAKAGE_OR_METADATA_COLUMNS))
available_drop_columns = [c for c in drop_columns if c in model_df.columns]
candidate_raw_features = [c for c in model_df.columns if c not in available_drop_columns]

meta_columns = [
    "row_id",
    "company_name",
    "role_title",
    "role_family",
    "seniority",
    "location",
    "applied",
    "source_type",
    "preference_reason",
]
meta = model_df[[c for c in meta_columns if c in model_df.columns]].copy()

display(pd.DataFrame({"dropped_from_training_X": available_drop_columns}))
display(pd.DataFrame({"candidate_raw_feature": candidate_raw_features}))

,dropped_from_training_X
0,applied
1,applied_clean
2,apply_url
3,company_match_score
4,company_name
5,company_name_standard
6,experience_profile_fit_score
7,location_profile_fit_score
8,near_duplicate_key
9,overall_preference_score


,candidate_raw_feature
0,can_apply_now
1,role_title
2,role_family
3,seniority
4,location
...,...
64,feat_onsite_or_hybrid_early_career
65,feat_bad_fit_combo
66,feat_strong_fit_combo
67,enhanced_title_skill_text


## Step 4: Feature Groups And Text Fields

In [6]:
engineered_feature_columns = [c for c in model_df.columns if c.startswith("feat_")]

base_numeric_features = [
    "experience_min_years",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "experience_profile_fit_imputed",
]

boolean_features = [
    "has_skills_or_requirements",
    "is_bengaluru",
    "is_gurgaon_ncr",
    "is_india",
    "salary_available",
]

categorical_features = [
    "can_apply_now",
    "freshness",
    "role_family_standard",
    "seniority_standard",
    "employment_type_standard",
    "work_mode_standard",
    "city_standard",
    "country_standard",
    "location_bucket",
    "experience_min_bucket",
    "skills_imputation_type",
    "experience_profile_fit_imputation_reason",
]

base_numeric_features = [c for c in base_numeric_features if c in model_df.columns]
boolean_features = [c for c in boolean_features if c in model_df.columns]
categorical_features = [c for c in categorical_features if c in model_df.columns]

for col in base_numeric_features + engineered_feature_columns:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")


def to_binary(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"true", "1", "yes"}:
        return 1.0
    if text in {"false", "0", "no"}:
        return 0.0
    return np.nan


for col in boolean_features:
    model_df[col] = model_df[col].map(to_binary)

for col in ["enhanced_title_skill_text", "enhanced_context_text"]:
    if col not in model_df.columns:
        model_df[col] = ""
    model_df[col] = model_df[col].fillna("").astype(str)

X_raw = model_df[candidate_raw_features].copy()
X_raw["enhanced_title_skill_text"] = model_df["enhanced_title_skill_text"]
X_raw["enhanced_context_text"] = model_df["enhanced_context_text"]

display(pd.DataFrame({
    "group": ["base_numeric", "boolean", "categorical", "engineered"],
    "count": [
        len(base_numeric_features),
        len(boolean_features),
        len(categorical_features),
        len(engineered_feature_columns),
    ],
    "columns": [
        ", ".join(base_numeric_features),
        ", ".join(boolean_features),
        ", ".join(categorical_features),
        ", ".join(engineered_feature_columns),
    ],
}))

,group,count,columns
0,base_numeric,5,"experience_min_years, salary_min, salary_max, salary_midpoint, experience_profile_fit_imputed"
1,boolean,5,"has_skills_or_requirements, is_bengaluru, is_gurgaon_ncr, is_india, salary_available"
2,categorical,12,"can_apply_now, freshness, role_family_standard, seniority_standard, employment_type_standard, work_mode_standard, city_standard, country..."
3,engineered,34,"feat_is_experience_mismatch, feat_is_senior_or_above, feat_is_lead_staff_principal, feat_is_early_career, feat_is_intern_or_new_grad, fe..."


## Step 5: Correlation-Based Feature Drop Functions

In [7]:
def safe_corr(x, y):
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    y = pd.to_numeric(pd.Series(y), errors="coerce")
    valid = x.notna() & y.notna()
    if valid.sum() < 3:
        return 0.0
    if x[valid].nunique() <= 1:
        return 0.0
    return float(x[valid].corr(y[valid]))


def select_low_correlation_features(X_train, y_train, candidate_cols, drop_count=20):
    rows = []
    for col in candidate_cols:
        if col not in X_train.columns:
            continue
        corr = safe_corr(X_train[col], y_train)
        rows.append({
            "feature": col,
            "correlation": corr,
            "abs_correlation": abs(corr),
            "non_null": int(pd.Series(X_train[col]).notna().sum()),
        })
    corr_df = pd.DataFrame(rows).sort_values(["abs_correlation", "feature"]).reset_index(drop=True)
    drop_features = corr_df.head(min(drop_count, len(corr_df)))["feature"].tolist()
    return drop_features, corr_df


global_drop_features, global_corr_df = select_low_correlation_features(
    X_raw,
    y_applied,
    engineered_feature_columns,
    drop_count=DROP_LOW_CORR_COUNT,
)

display(global_corr_df.sort_values("abs_correlation", ascending=False).head(20))
display(global_corr_df.head(DROP_LOW_CORR_COUNT))

,feature,correlation,abs_correlation,non_null
33,feat_is_early_career,0.634859,0.634859,691
32,feat_strong_fit_combo,0.607620,0.607620,691
31,feat_is_experience_mismatch,-0.574479,0.574479,691
30,feat_onsite_or_hybrid_early_career,0.515850,0.515850,691
29,feat_salary_good_but_role_bad,-0.420000,0.420000,691
28,feat_bad_fit_combo,-0.410917,0.410917,691
27,feat_is_entry_or_junior,0.404016,0.404016,691
26,feat_is_high_salary,-0.398678,0.398678,691
25,feat_is_senior_or_above,-0.397013,0.397013,691
24,feat_good_title_bad_experience,-0.396822,0.396822,691


,feature,correlation,abs_correlation,non_null
0,feat_is_support_or_annotation,-0.020366,0.020366,691
1,feat_is_qa_testing,0.027270,0.027270,691
2,feat_is_platform_infra,-0.031317,0.031317,691
3,feat_is_india_remote,-0.035138,0.035138,691
4,feat_is_data_role,0.058829,0.058829,691
5,feat_is_security_role,-0.073803,0.073803,691
6,feat_is_ml_ai,-0.090918,0.090918,691
7,feat_is_backend_or_fullstack,0.093927,0.093927,691
8,feat_is_remote_global,-0.126539,0.126539,691
9,feat_is_low_salary,0.146707,0.146707,691


## Step 6: Pipelines With Better NLP

In [8]:
def make_preprocess(extra_numeric_features=None, dropped_engineered_features=None):
    extra_numeric_features = list(extra_numeric_features or [])
    dropped_engineered_features = set(dropped_engineered_features or [])

    active_engineered = [c for c in engineered_feature_columns if c not in dropped_engineered_features]
    numeric_cols = base_numeric_features + boolean_features + active_engineered + extra_numeric_features

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=3)),
    ])

    title_skill_word = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=2600,
        sublinear_tf=True,
    )

    context_char = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=1800,
        sublinear_tf=True,
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_cols),
            ("categorical", categorical_transformer, categorical_features),
            ("title_skill_word", title_skill_word, "enhanced_title_skill_text"),
            ("context_char", context_char, "enhanced_context_text"),
        ],
        remainder="drop",
    )


def make_score_model(alpha=2.0):
    return Pipeline(steps=[
        ("preprocess", make_preprocess(dropped_engineered_features=[])),
        ("regressor", Ridge(alpha=alpha)),
    ])


def make_xgb_model(extra_numeric_features=None, dropped_engineered_features=None, scale_pos_weight=1.0):
    return Pipeline(steps=[
        ("preprocess", make_preprocess(
            extra_numeric_features=extra_numeric_features,
            dropped_engineered_features=dropped_engineered_features,
        )),
        ("classifier", XGBClassifier(
            n_estimators=450,
            max_depth=3,
            learning_rate=0.03,
            min_child_weight=4,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.15,
            reg_lambda=3.5,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=4,
            scale_pos_weight=scale_pos_weight,
        )),
    ])


SCORE_BOUNDS = {"salary_profile_fit_score": (0.0, 8.0)}


def clip_scores(predictions):
    predictions = np.asarray(predictions, dtype=float).copy()
    for idx, target in enumerate(SCORE_TARGETS):
        low, high = SCORE_BOUNDS.get(target, (0.0, 5.0))
        predictions[:, idx] = np.clip(predictions[:, idx], low, high)
    return predictions


PRED_SCORE_COLUMNS = ["pred_" + c for c in SCORE_TARGETS]
TRUE_SCORE_COLUMNS = ["true_" + c for c in SCORE_TARGETS]


def append_score_columns(X_base, score_array, prefix):
    X_aug = X_base.copy()
    score_array = clip_scores(score_array)
    for idx, score_name in enumerate(SCORE_TARGETS):
        X_aug[prefix + "_" + score_name] = score_array[:, idx]
    return X_aug

## Step 7: Metrics

In [9]:
def regression_metrics(y_true, y_pred, split_name, fold=None):
    y_true_df = pd.DataFrame(np.asarray(y_true, dtype=float), columns=SCORE_TARGETS)
    y_pred_df = pd.DataFrame(clip_scores(y_pred), columns=SCORE_TARGETS)
    rows = []
    for target in SCORE_TARGETS:
        actual = y_true_df[target]
        pred = y_pred_df[target]
        abs_error = (pred - actual).abs()
        rows.append({
            "fold": fold,
            "split": split_name,
            "target": target,
            "mae": mean_absolute_error(actual, pred),
            "rmse": np.sqrt(np.mean(np.square(pred - actual))),
            "r2": r2_score(actual, pred),
            "spearman": actual.corr(pred, method="spearman"),
            "within_0_5": (abs_error <= 0.5).mean(),
            "within_1_0": (abs_error <= 1.0).mean(),
        })
    return pd.DataFrame(rows)


def classification_summary(model_name, fold, y_true, proba, threshold=0.50):
    proba = np.asarray(proba, dtype=float)
    pred = (proba >= threshold).astype(int)
    return {
        "model": model_name,
        "fold": fold,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_yes": precision_score(y_true, pred, zero_division=0),
        "recall_yes": recall_score(y_true, pred, zero_division=0),
        "f1_yes": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "average_precision": average_precision_score(y_true, proba),
        "log_loss": log_loss(y_true, proba, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, proba),
        "predicted_yes_count": int(pred.sum()),
    }


def ranking_summary(model_name, y_true, proba, ks=(10, 20, 50)):
    frame = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(proba)})
    frame = frame.sort_values("p", ascending=False).reset_index(drop=True)
    rows = []
    total_pos = max(int(frame["y"].sum()), 1)
    for k in ks:
        top = frame.head(k)
        precision_at_k = top["y"].mean()
        recall_at_k = top["y"].sum() / total_pos
        gains = top["y"].values
        discounts = 1 / np.log2(np.arange(2, len(gains) + 2))
        dcg = float(np.sum(gains * discounts))
        ideal_gains = np.sort(frame["y"].values)[::-1][:k]
        ideal_dcg = float(np.sum(ideal_gains * discounts[:len(ideal_gains)]))
        rows.append({
            "model": model_name,
            "k": k,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k,
            "ndcg_at_k": dcg / ideal_dcg if ideal_dcg else 0.0,
        })
    return pd.DataFrame(rows)


def threshold_table(y_true, proba, model_name):
    rows = []
    for threshold in np.arange(0.20, 0.81, 0.05):
        pred = (np.asarray(proba) >= threshold).astype(int)
        rows.append({
            "model": model_name,
            "threshold": round(float(threshold), 2),
            "predicted_yes_count": int(pred.sum()),
            "precision_yes": precision_score(y_true, pred, zero_division=0),
            "recall_yes": recall_score(y_true, pred, zero_division=0),
            "f1_yes": f1_score(y_true, pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        })
    return pd.DataFrame(rows)

## Step 8: Leakage-Safe K-Fold

In [10]:
def run_enhanced_stacked_xgb_cv():
    outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    classification_rows = []
    regression_rows = []
    prediction_frames = []
    fold_drop_rows = []

    for fold_idx, (train_idx, val_idx) in enumerate(outer_cv.split(X_raw, y_applied), start=1):
        X_tr_raw = X_raw.iloc[train_idx].copy()
        X_val_raw = X_raw.iloc[val_idx].copy()
        y_tr = y_applied.iloc[train_idx].copy()
        y_val = y_applied.iloc[val_idx].copy()
        y_tr_scores = y_scores.iloc[train_idx].copy()
        y_val_scores = y_scores.iloc[val_idx].copy()

        dropped_features, fold_corr = select_low_correlation_features(
            X_tr_raw,
            y_tr,
            engineered_feature_columns,
            drop_count=DROP_LOW_CORR_COUNT,
        )
        for _, row in fold_corr.iterrows():
            fold_drop_rows.append({
                "fold": fold_idx,
                "feature": row["feature"],
                "correlation": row["correlation"],
                "abs_correlation": row["abs_correlation"],
                "dropped": row["feature"] in dropped_features,
            })

        scale_pos_weight = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))

        fold_score_model = make_score_model(alpha=2.0)
        train_score_oof = cross_val_predict(
            fold_score_model,
            X_tr_raw,
            y_tr_scores,
            cv=inner_cv,
            n_jobs=None,
        )
        train_score_oof = clip_scores(train_score_oof)
        fold_score_model.fit(X_tr_raw, y_tr_scores)
        val_score_pred = clip_scores(fold_score_model.predict(X_val_raw))

        regression_rows.append(regression_metrics(y_tr_scores, train_score_oof, "train_inner_oof", fold=fold_idx))
        regression_rows.append(regression_metrics(y_val_scores, val_score_pred, "outer_validation", fold=fold_idx))

        X_tr_stacked = append_score_columns(X_tr_raw, train_score_oof, prefix="pred")
        X_val_stacked = append_score_columns(X_val_raw, val_score_pred, prefix="pred")
        X_tr_oracle = append_score_columns(X_tr_raw, y_tr_scores.values, prefix="true")
        X_val_oracle = append_score_columns(X_val_raw, y_val_scores.values, prefix="true")

        fold_configs = [
            ("raw_enhanced_xgboost", make_xgb_model(dropped_engineered_features=dropped_features, scale_pos_weight=scale_pos_weight), X_tr_raw, X_val_raw),
            ("stacked_ridge_scores_enhanced_xgboost", make_xgb_model(PRED_SCORE_COLUMNS, dropped_engineered_features=dropped_features, scale_pos_weight=scale_pos_weight), X_tr_stacked, X_val_stacked),
            ("oracle_true_scores_enhanced_xgboost", make_xgb_model(TRUE_SCORE_COLUMNS, dropped_engineered_features=dropped_features, scale_pos_weight=scale_pos_weight), X_tr_oracle, X_val_oracle),
        ]

        fold_predictions = meta.iloc[val_idx].reset_index(drop=True).copy()
        fold_predictions["fold"] = fold_idx
        fold_predictions["actual_applied"] = y_val.reset_index(drop=True).values

        for target_idx, target in enumerate(SCORE_TARGETS):
            fold_predictions["true_" + target] = y_val_scores.reset_index(drop=True)[target]
            fold_predictions["pred_" + target] = val_score_pred[:, target_idx]

        for model_name, estimator, X_tr, X_val in fold_configs:
            estimator.fit(X_tr, y_tr)
            proba = estimator.predict_proba(X_val)[:, 1]
            classification_rows.append(classification_summary(model_name, fold_idx, y_val, proba, threshold=0.50))
            fold_predictions[model_name + "_probability"] = proba
            fold_predictions[model_name + "_prediction"] = (proba >= 0.50).astype(int)

        prediction_frames.append(fold_predictions)

    return (
        pd.DataFrame(classification_rows),
        pd.concat(regression_rows, ignore_index=True),
        pd.concat(prediction_frames, ignore_index=True).sort_values(["fold", "row_id"]).reset_index(drop=True),
        pd.DataFrame(fold_drop_rows),
    )


cv_detail, score_cv_detail, oof_predictions, fold_feature_correlations = run_enhanced_stacked_xgb_cv()

display(cv_detail)
display(score_cv_detail.head(20))
display(fold_feature_correlations.head(20))
display(oof_predictions.head(10))

,model,fold,threshold,accuracy,balanced_accuracy,precision_yes,recall_yes,f1_yes,roc_auc,average_precision,log_loss,brier_score,predicted_yes_count
0,raw_enhanced_xgboost,1,0.5,0.913669,0.908785,0.929825,0.868852,0.898305,0.943674,0.942833,0.288381,0.082404,57
1,stacked_ridge_scores_enhanced_xgboost,1,0.5,0.913669,0.908785,0.929825,0.868852,0.898305,0.942203,0.942216,0.296667,0.085681,57
2,oracle_true_scores_enhanced_xgboost,1,0.5,0.906475,0.905948,0.887097,0.901639,0.894309,0.943674,0.946013,0.294092,0.080569,62
3,raw_enhanced_xgboost,2,0.5,0.876812,0.870875,0.892857,0.819672,0.854701,0.930381,0.914174,0.340620,0.097509,56
4,stacked_ridge_scores_enhanced_xgboost,2,0.5,0.898551,0.895465,0.898305,0.868852,0.883333,0.938046,0.920988,0.318220,0.088046,59
5,oracle_true_scores_enhanced_xgboost,2,0.5,0.876812,0.874281,0.866667,0.852459,0.859504,0.941878,0.923686,0.301357,0.084509,60
6,raw_enhanced_xgboost,3,0.5,0.869565,0.872898,0.820896,0.901639,0.859375,0.918671,0.873340,0.381916,0.108842,67
7,stacked_ridge_scores_enhanced_xgboost,3,0.5,0.855072,0.854801,0.825397,0.852459,0.838710,0.918459,0.879157,0.402779,0.115189,63
8,oracle_true_scores_enhanced_xgboost,3,0.5,0.891304,0.894081,0.848485,0.918033,0.881890,0.930594,0.889014,0.343123,0.092776,66
9,raw_enhanced_xgboost,4,0.5,0.905797,0.901959,0.913793,0.868852,0.890756,0.965936,0.964892,0.237541,0.071906,58


,fold,split,target,mae,rmse,r2,spearman,within_0_5,within_1_0
0,1,train_inner_oof,role_title_preference_score,0.397003,0.692009,0.668084,0.768913,0.744565,0.882246
1,1,train_inner_oof,role_family_preference_score,0.297279,0.567768,0.700389,0.814155,0.820652,0.932971
2,1,train_inner_oof,seniority_profile_fit_score,0.221894,0.419746,0.923596,0.930744,0.875000,0.956522
3,1,train_inner_oof,location_profile_fit_score,0.310186,0.464290,0.907806,0.888926,0.780797,0.947464
4,1,train_inner_oof,work_mode_fit_score,0.270979,0.435195,0.769763,0.824923,0.827899,0.952899
5,1,train_inner_oof,experience_profile_fit_score,0.135252,0.257323,0.977080,0.908869,0.934783,0.990942
6,1,train_inner_oof,skills_profile_fit_score,0.334437,0.582644,0.765367,0.839284,0.780797,0.913043
7,1,train_inner_oof,salary_profile_fit_score,0.381989,0.482850,0.919390,0.946885,0.730072,0.956522
8,1,train_inner_oof,overall_preference_score,0.508666,0.657139,0.810503,0.863220,0.567029,0.909420
9,1,outer_validation,role_title_preference_score,0.455309,0.719115,0.683335,0.848637,0.640288,0.884892


,fold,feature,correlation,abs_correlation,dropped
0,1,feat_is_qa_testing,0.009973,0.009973,True
1,1,feat_is_platform_infra,-0.017341,0.017341,True
2,1,feat_is_support_or_annotation,-0.033035,0.033035,True
3,1,feat_is_india_remote,-0.035862,0.035862,True
4,1,feat_is_data_role,0.043253,0.043253,True
5,1,feat_is_security_role,-0.058669,0.058669,True
6,1,feat_is_ml_ai,-0.113448,0.113448,True
7,1,feat_remote_early_career,0.127162,0.127162,True
8,1,feat_is_low_salary,0.137873,0.137873,True
9,1,feat_is_backend_or_fullstack,0.152028,0.152028,True


,row_id,company_name,role_title,role_family,seniority,location,applied,source_type,preference_reason,fold,actual_applied,true_role_title_preference_score,pred_role_title_preference_score,true_role_family_preference_score,pred_role_family_preference_score,true_seniority_profile_fit_score,pred_seniority_profile_fit_score,true_location_profile_fit_score,pred_location_profile_fit_score,true_work_mode_fit_score,pred_work_mode_fit_score,true_experience_profile_fit_score,pred_experience_profile_fit_score,true_skills_profile_fit_score,pred_skills_profile_fit_score,true_salary_profile_fit_score,pred_salary_profile_fit_score,true_overall_preference_score,pred_overall_preference_score,raw_enhanced_xgboost_probability,raw_enhanced_xgboost_prediction,stacked_ridge_scores_enhanced_xgboost_probability,stacked_ridge_scores_enhanced_xgboost_prediction,oracle_true_scores_enhanced_xgboost_probability,oracle_true_scores_enhanced_xgboost_prediction
0,14,PhonePe,"Software Engineer, Android",Frontend / Mobile,junior/mid,"Pune, India",yes,deterministic_scrape_candidate,Great company; mobile not primary profile unless you want Android exposure,1,1,5.0,4.149940,5.0,4.720085,5.0,4.382609,4.0,4.014601,5.0,5.000000,5.0,4.012412,5.0,4.222762,6.0,5.553412,5.0,3.844254,0.699101,1,0.746129,1,0.945864,1
1,15,Addepar,"Software Engineer, Backend – AI Platform",Backend / AI Platform,mid,"Pune, India",yes,deterministic_scrape_candidate,Strong AI/backend/platform fit; likely experience stretch,1,1,5.0,2.961650,5.0,4.361262,3.0,2.293164,4.0,3.709623,5.0,5.000000,2.0,1.923322,4.0,3.401251,7.0,6.850238,3.0,2.315576,0.409561,0,0.379362,0,0.175493,0
2,22,Perch Energy,Software Engineer III,Backend / Fullstack,mid,Remote India,no,deterministic_scrape_candidate,Remote India fullstack/backend but level III likely above fresher,1,0,2.0,4.238052,5.0,4.931303,2.0,2.582269,5.0,4.461513,5.0,5.000000,1.0,2.065515,4.0,3.887536,5.0,4.849779,1.0,2.753440,0.355602,0,0.302360,0,0.183643,0
3,23,Truveta,Senior Software Engineer (Full-Stack),Fullstack,senior,"Hyderabad, India",no,deterministic_scrape_candidate,Fullstack fit but seniority too high for current profile,1,0,3.0,3.154099,5.0,4.688288,2.0,1.708410,3.0,3.431191,5.0,5.000000,0.0,0.180044,5.0,4.677407,6.0,5.742253,0.0,1.148345,0.034954,0,0.054459,0,0.024582,0
4,26,SonicWall,Full Stack Software Engineer - Networking exp is a must,Fullstack / Networking,mid,"Pune, Maharashtra, India",no,deterministic_scrape_candidate,"Good fullstack/cybersecurity, but networking experience is explicitly required",1,0,3.0,3.895451,3.0,5.000000,2.0,2.678481,4.0,3.761323,5.0,5.000000,1.0,2.097907,4.0,4.385226,5.0,5.068009,0.0,2.212877,0.157439,0,0.122362,0,0.068485,0
5,30,LivePerson,Software Development Engineer III - Node JS/Python,Backend / Generative AI,senior,India,no,deterministic_scrape_candidate,"JEDI-like GenAI/RAG stack, but senior/SDE III level",1,0,3.0,4.559796,5.0,4.937864,0.0,1.773786,5.0,4.189267,5.0,4.954318,0.0,0.509667,5.0,4.351319,5.0,5.251099,0.0,1.139950,0.257026,0,0.108475,0,0.117845,0
6,33,Fluxon,Senior Software Engineer (AI Engineer),Generative AI / Backend,senior,India/Remote likely,no,deterministic_scrape_candidate,"Excellent JEDI/resume fit, but senior role",1,0,2.0,3.037029,4.0,4.138724,2.0,1.304296,5.0,4.035406,5.0,4.678304,0.0,0.380891,4.0,3.217862,6.0,5.658502,0.0,1.221023,0.055296,0,0.065059,0,0.039744,0
7,36,Sumo Logic,Senior Machine Learning Engineer,AI/ML / Observability,senior,"Bengaluru, Karnataka, India",no,deterministic_scrape_candidate,"Strong AI-agent domain, but senior ML role",1,0,3.0,2.252339,3.0,3.153507,1.0,1.195907,4.0,4.713588,5.0,4.972743,0.0,0.062817,3.0,3.021811,7.0,7.043841,2.0,1.092936,0.006889,0,0.012268,0,0.005550,0
8,38,Twilio,Staff Machine Learning Engineer (L4),AI/ML / LLM,staff,Remote - India,no,deterministic_scrape_candidate,"Remote India and LLM relevant, but staff level is too senior",1,0,1.0,2.921332,4.0,3.646267,1.0,0.273075,4.0,4.327100,5.0,4.399713,0.0,0.000000,3.0,2.766756,8.0,8.000000,

## Step 9: Aggregate Results

In [11]:
metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision_yes",
    "recall_yes",
    "f1_yes",
    "roc_auc",
    "average_precision",
    "log_loss",
    "brier_score",
    "predicted_yes_count",
]

cv_summary = cv_detail.groupby("model")[metric_columns].agg(["mean", "std"])
flat_cv_summary = cv_summary.copy()
flat_cv_summary.columns = [metric + "_" + stat for metric, stat in flat_cv_summary.columns]
flat_cv_summary = flat_cv_summary.reset_index().sort_values("f1_yes_mean", ascending=False)

display(cv_summary)
display(flat_cv_summary)

score_cv_summary = (
    score_cv_detail
    .groupby(["split", "target"])[["mae", "rmse", "r2", "spearman", "within_0_5", "within_1_0"]]
    .agg(["mean", "std"])
)
display(score_cv_summary)

accuracy           balanced_accuracy  \
                                           mean       std              mean   
model                                                                         
oracle_true_scores_enhanced_xgboost    0.887092  0.026050          0.887236   
raw_enhanced_xgboost                   0.879835  0.031999          0.876605   
stacked_ridge_scores_enhanced_xgboost  0.879835  0.030744          0.876264   

                                                precision_yes            \
                                            std          mean       std   
model                                                                     
oracle_true_scores_enhanced_xgboost    0.025282      0.861384  0.038344   
raw_enhanced_xgboost                   0.031774      0.876991  0.049963   
stacked_ridge_scores_enhanced_xgboost  0.030458      0.878437  0.043292   

                                      recall_yes              f1_yes  \
                                            mean       std      mean   
model                                                                  
oracle_true_scores_enhanced_xgboost     0.888525  0.026937  0.874398   
raw_enhanced_xgboost                    0.849180  0.045490  0.861972   
stacked_ridge_scores_enhanced_xgboost   0.845902  0.033994  0.861571   

                                                  roc_auc            \
                                            std      mean       std   
model                                                                 
oracle_true_scores_enhanced_xgboost    0.027371  0.943727  0.014253   
raw_enhanced_xgboost                   0.036265  0.933891  0.021794   
stacked_ridge_scores_enhanced_xgboost  0.034655  0.935513  0.021754   

                                      average_precision            log_loss  \
                                                   mean       std      mean   
model                                                                         
oracle_true_scores_enhanced_xgboost            0.925516  0.030579  0.301153   
raw_enhanced_xgboost                           0.914546  0.040019  0.330946   
stacked_ridge_scores_enhanced_xgboost          0.915303  0.040842  0.334187   

                                                brier_score            \
                                            std        mean       std   
model                                                                   
oracle_true_scores_enhanced_xgboost    0.046402    0.085910  0.013683   
raw_enhanced_xgboost                   0.068742    0.096744  0.020384   
stacked_ridge_scores_enhanced_xgboost  0.074293    0.097421  0.021483   

                                      predicted_yes_count            
                                                     mean       std  
model                                                                
oracle_true_scores_enhanced_xgboost                  63.0  2.828427  
raw_enhanced_xgboost                                 59.2  4.438468  
stacked_ridge_scores_enhanced_xgboost                58.8  2.489980

,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_yes_mean,precision_yes_std,recall_yes_mean,recall_yes_std,f1_yes_mean,f1_yes_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,log_loss_mean,log_loss_std,brier_score_mean,brier_score_std,predicted_yes_count_mean,predicted_yes_count_std
0,oracle_true_scores_enhanced_xgboost,0.887092,0.026050,0.887236,0.025282,0.861384,0.038344,0.888525,0.026937,0.874398,0.027371,0.943727,0.014253,0.925516,0.030579,0.301153,0.046402,0.085910,0.013683,63.0,2.828427
1,raw_enhanced_xgboost,0.879835,0.031999,0.876605,0.031774,0.876991,0.049963,0.849180,0.045490,0.861972,0.036265,0.933891,0.021794,0.914546,0.040019,0.330946,0.068742,0.096744,0.020384,59.2,4.438468
2,stacked_ridge_scores_enhanced_xgboost,0.879835,0.030744,0.876264,0.030458,0.878437,0.043292,0.845902,0.033994,0.861571,0.034655,0.935513,0.021754,0.915303,0.040842,0.334187,0.074293,0.097421,0.021483,58.8,2.489980


mae                rmse  \
                                                   mean       std      mean   
split            target                                                       
outer_validation experience_profile_fit_score  0.121395  0.020250  0.245929   
                 location_profile_fit_score    0.296004  0.024344  0.444343   
                 overall_preference_score      0.504391  0.015237  0.638989   
                 role_family_preference_score  0.270553  0.046027  0.488560   
                 role_title_preference_score   0.396962  0.045031  0.662510   
                 salary_profile_fit_score      0.372398  0.015125  0.469543   
                 seniority_profile_fit_score   0.200885  0.016307  0.370237   
                 skills_profile_fit_score      0.303468  0.022904  0.534933   
                 work_mode_fit_score           0.242059  0.025501  0.391820   
train_inner_oof  experience_profile_fit_score  0.134995  0.009179  0.259455   
                 location_profile_fit_score    0.301769  0.011550  0.456551   
                 overall_preference_score      0.506254  0.003504  0.650236   
                 role_family_preference_score  0.291446  0.016240  0.539935   
                 role_title_preference_score   0.409937  0.016279  0.688989   
                 salary_profile_fit_score      0.379413  0.005102  0.478841   
                 seniority_profile_fit_score   0.214187  0.007089  0.387301   
                 skills_profile_fit_score      0.336410  0.013447  0.591986   
                 work_mode_fit_score           0.266675  0.008574  0.431457   

                                                               r2            \
                                                    std      mean       std   
split            target                                                       
outer_validation experience_profile_fit_score  0.043604  0.978555  0.006574   
                 location_profile_fit_score    0.040691  0.907337  0.026328   
                 overall_preference_score      0.033294  0.818649  0.019920   
                 role_family_preference_score  0.102786  0.779543  0.067534   
                 role_title_preference_score   0.048019  0.702214  0.014566   
                 salary_profile_fit_score      0.018182  0.922503  0.007441   
                 seniority_profile_fit_score   0.017302  0.941230  0.006672   
                 skills_profile_fit_score      0.054721  0.797345  0.042983   
                 work_mode_fit_score           0.034888  0.803435  0.015897   
train_inner_oof  experience_profile_fit_score  0.011739  0.976763  0.001872   
                 location_profile_fit_score    0.016603  0.906525  0.010174   
                 overall_preference_score      0.007710  0.813274  0.003600   
                 role_family_preference_score  0.034255  0.734321  0.028369   
                 role_title_preference_score   0.018307  0.679955  0.015559   
                 salary_profile_fit_score      0.006757  0.919988  0.001732   
                 seniority_profile_fit_score   0.024183  0.935902  0.008633   
                 skills_profile_fit_score      0.026775  0.756078  0.020782   
                 work_mode_fit_score           0.010540  0.767840  0.006002   

                                               spearman           within_0_5  \
                                                   mean       std       mean   
split            target                                                        
outer_validation experience_profile_fit_score  0.909111  0.021052   0.934856   
                 location_profile_fit_score    0.888884  0.013404   0.795975   
                 overall_preference_score      0.873979  0.012377   0.562955   
                 role_family_preference_score  0.836697  0.020318   0.816213   
                 role_title_preference_score   0.804969  0.027120   0.700521   
                 salary_profile_fit_score      0.949747  0.006219   0.733698   
            

## Step 10: Threshold And Ranking Metrics

In [12]:
threshold_review = pd.concat([
    threshold_table(oof_predictions["actual_applied"], oof_predictions[model + "_probability"], model)
    for model in [
        "raw_enhanced_xgboost",
        "stacked_ridge_scores_enhanced_xgboost",
        "oracle_true_scores_enhanced_xgboost",
    ]
], ignore_index=True)

ranking_review = pd.concat([
    ranking_summary(model, oof_predictions["actual_applied"], oof_predictions[model + "_probability"])
    for model in [
        "raw_enhanced_xgboost",
        "stacked_ridge_scores_enhanced_xgboost",
        "oracle_true_scores_enhanced_xgboost",
    ]
], ignore_index=True)

display(threshold_review.sort_values(["model", "f1_yes"], ascending=[True, False]).groupby("model").head(5))
display(ranking_review)

,model,threshold,predicted_yes_count,precision_yes,recall_yes,f1_yes,balanced_accuracy
29,oracle_true_scores_enhanced_xgboost,0.35,333,0.846847,0.924590,0.884013,0.896233
31,oracle_true_scores_enhanced_xgboost,0.45,322,0.857143,0.904918,0.880383,0.892874
30,oracle_true_scores_enhanced_xgboost,0.40,327,0.850153,0.911475,0.879747,0.892266
28,oracle_true_scores_enhanced_xgboost,0.30,351,0.817664,0.940984,0.875000,0.887590
32,oracle_true_scores_enhanced_xgboost,0.50,315,0.860317,0.888525,0.874194,0.887267
6,raw_enhanced_xgboost,0.50,296,0.875000,0.849180,0.861897,0.876663
5,raw_enhanced_xgboost,0.45,310,0.851613,0.865574,0.858537,0.873201
3,raw_enhanced_xgboost,0.35,338,0.813609,0.901639,0.855365,0.869213
4,raw_enhanced_xgboost,0.40,315,0.841270,0.868852,0.854839,0.869659
7,raw_enhanced_xgboost,0.55,288,0.878472,0.829508,0.853288,0.869417


,model,k,precision_at_k,recall_at_k,ndcg_at_k
0,raw_enhanced_xgboost,10,1.00,0.032787,1.000000
1,raw_enhanced_xgboost,20,1.00,0.065574,1.000000
2,raw_enhanced_xgboost,50,0.96,0.157377,0.970952
3,stacked_ridge_scores_enhanced_xgboost,10,1.00,0.032787,1.000000
4,stacked_ridge_scores_enhanced_xgboost,20,1.00,0.065574,1.000000
5,stacked_ridge_scores_enhanced_xgboost,50,0.94,0.154098,0.955664
6,oracle_true_scores_enhanced_xgboost,10,1.00,0.032787,1.000000
7,oracle_true_scores_enhanced_xgboost,20,1.00,0.065574,1.000000
8,oracle_true_scores_enhanced_xgboost,50,0.98,0.160656,0.986191


## Step 11: Save Artifacts

In [13]:
cv_detail_path = OUTPUT_DIR / "jedi_enhanced_xgboost_outer_cv_detail.csv"
cv_summary_path = OUTPUT_DIR / "jedi_enhanced_xgboost_outer_cv_summary.csv"
score_cv_detail_path = OUTPUT_DIR / "jedi_enhanced_xgboost_ridge_score_cv_detail.csv"
oof_predictions_path = OUTPUT_DIR / "jedi_enhanced_xgboost_oof_predictions.csv"
threshold_review_path = OUTPUT_DIR / "jedi_enhanced_xgboost_threshold_review.csv"
ranking_review_path = OUTPUT_DIR / "jedi_enhanced_xgboost_ranking_review.csv"
fold_feature_corr_path = OUTPUT_DIR / "jedi_enhanced_xgboost_fold_feature_correlations.csv"

cv_detail.to_csv(cv_detail_path, index=False)
flat_cv_summary.to_csv(cv_summary_path, index=False)
score_cv_detail.to_csv(score_cv_detail_path, index=False)
oof_predictions.to_csv(oof_predictions_path, index=False)
threshold_review.to_csv(threshold_review_path, index=False)
ranking_review.to_csv(ranking_review_path, index=False)
fold_feature_correlations.to_csv(fold_feature_corr_path, index=False)

display(pd.DataFrame({
    "artifact": [
        "cv_detail",
        "cv_summary",
        "ridge_score_cv_detail",
        "oof_predictions",
        "threshold_review",
        "ranking_review",
        "fold_feature_correlations",
    ],
    "path": [
        str(cv_detail_path),
        str(cv_summary_path),
        str(score_cv_detail_path),
        str(oof_predictions_path),
        str(threshold_review_path),
        str(ranking_review_path),
        str(fold_feature_corr_path),
    ],
}))

,artifact,path
0,cv_detail,d:\ML\JEDI\data\jedi_enhanced_xgboost_outer_cv_detail.csv
1,cv_summary,d:\ML\JEDI\data\jedi_enhanced_xgboost_outer_cv_summary.csv
2,ridge_score_cv_detail,d:\ML\JEDI\data\jedi_enhanced_xgboost_ridge_score_cv_detail.csv
3,oof_predictions,d:\ML\JEDI\data\jedi_enhanced_xgboost_oof_predictions.csv
4,threshold_review,d:\ML\JEDI\data\jedi_enhanced_xgboost_threshold_review.csv
5,ranking_review,d:\ML\JEDI\data\jedi_enhanced_xgboost_ranking_review.csv
6,fold_feature_correlations,d:\ML\JEDI\data\jedi_enhanced_xgboost_fold_feature_correlations.csv


## Step 12: Readout

In [14]:
best_model = flat_cv_summary.iloc[0]["model"]
best_f1 = flat_cv_summary.iloc[0]["f1_yes_mean"]
best_auc = flat_cv_summary.iloc[0]["roc_auc_mean"]

print("Best CV model by F1_yes:", best_model)
print("F1_yes mean: {:.4f}".format(best_f1))
print("ROC-AUC mean: {:.4f}".format(best_auc))

display(
    oof_predictions
    .sort_values("stacked_ridge_scores_enhanced_xgboost_probability", ascending=False)
    .head(25)
    [[
        "row_id",
        "company_name",
        "role_title",
        "actual_applied",
        "stacked_ridge_scores_enhanced_xgboost_probability",
        "raw_enhanced_xgboost_probability",
        "oracle_true_scores_enhanced_xgboost_probability",
    ] + ["pred_" + c for c in SCORE_TARGETS]]
)

Best CV model by F1_yes: oracle_true_scores_enhanced_xgboost
F1_yes mean: 0.8744
ROC-AUC mean: 0.9437


,row_id,company_name,role_title,actual_applied,stacked_ridge_scores_enhanced_xgboost_probability,raw_enhanced_xgboost_probability,oracle_true_scores_enhanced_xgboost_probability,pred_role_title_preference_score,pred_role_family_preference_score,pred_seniority_profile_fit_score,pred_location_profile_fit_score,pred_work_mode_fit_score,pred_experience_profile_fit_score,pred_skills_profile_fit_score,pred_salary_profile_fit_score,pred_overall_preference_score
81,391,JumpCloud India,Junior Full Stack Developer - Data Products,1,0.997224,0.992928,0.996374,5.000000,5.000000,5.000000,4.107462,5.000000,4.977750,4.890856,4.251224,4.841415
214,350,TerraPay,Junior Backend Developer - Product Team,1,0.996581,0.995808,0.996538,5.000000,4.188018,5.000000,4.956203,4.915377,5.000000,4.011417,4.321134,4.958750
83,397,Stripe,Junior Backend Developer - Product Team,1,0.996091,0.993318,0.994926,4.968755,4.145000,4.974439,3.974015,4.996367,4.939443,3.828766,4.181366,4.611606
301,172,360customizer,Backend Engineer,1,0.995561,0.994982,0.994238,4.937711,4.954174,5.000000,3.897428,4.371130,5.000000,3.809506,4.135301,4.479229
492,393,Lokal App,Junior Backend Developer - Growth Platform,1,0.995504,0.994796,0.993988,4.926164,4.038314,5.000000,4.904661,4.129013,5.000000,3.897742,4.104793,4.614286
345,375,Coupa India,Junior Backend Developer,1,0.995477,0.993710,0.994986,5.000000,4.145006,5.000000,4.936132,4.898588,4.983309,3.921391,4.228224,4.793611
626,395,Postman,Junior Full Stack Developer - Data Products,1,0.995459,0.990642,0.992501,5.000000,5.000000,4.942980,4.803105,4.971318,4.887881,5.000000,4.255286,4.516062
356,418,Coupa India,Junior Backend Developer - Marketplace,1,0.995251,0.992022,0.993371,5.000000,4.153266,5.000000,3.994288,4.882241,4.980059,3.999228,4.460110,4.615204
358,423,Gyrus.AI,Junior Full Stack Developer - SaaS Platform,1,0.995051,0.991532,0.995568,4.850451,4.937771,4.996014,4.072762,5.000000,4.963421,5.000000,4.479089,4.662536
491,371,Stripe,SDE I - Full Stack Engineer - Growth Platform,1,0.994974,0.990151,0.990273,5.000000,4.968767,4.956310,4.658491,4.872456,5.000000,5.000000,4.461183,4.540738
